# 08: Train on any tabular CSV (ADR-010)

This notebook takes **any CSV with a header row** and a column you want to predict, and produces a model folder (`model.skops` + `metadata.json`) that the API and web page serve **without any code change**.

It supports three kinds of target, inferred from the column unless you say otherwise:

| Target looks like | Task | Selected by (cross-validation, training split only) |
|---|---|---|
| two values (Yes/No, 0/1, fraud/ok) | `binary` | F1 of the positive class |
| 3 to 50 classes, or integer codes with few values | `multiclass` | macro F1 |
| a number with many distinct values | `regression` | RMSE (lower is better) |

Same rules as the Telco work: split first, all preprocessing inside one Pipeline fitted on the training split, the test split scored **once**.

**What you must do:** put your CSV in `MyDrive/ai-experiment-lab/data/`, set `DATA_FILE` and `TARGET` in step 1, then *Run all*.

**Limits (honest):** tabular CSV only (no images or free text); text columns with more than 50 distinct values become free-text inputs; dates are dropped; the serving image must use the same scikit-learn (and XGBoost, if the winner is XGBoost) version as this notebook (step 6 prints what to set).

In [ ]:
# --- AI Experiment Lab: standard Colab setup cell ---------------------------------
import os, subprocess, sys

REPO_URL = "https://github.com/sudhanshu230230/AI_Experiment_Lab.git"   # <-- edit
REPO_DIR = "/content/ai-experiment-lab"
DRIVE_ROOT = "/content/drive/MyDrive/ai-experiment-lab"
EXTRAS = "ml,tracking"          # pip extras from pyproject.toml

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/mlflow", exist_ok=True)

# For a private repo, store a token in Colab Secrets as GITHUB_TOKEN (never paste it here).
url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", url, REPO_DIR], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[{EXTRAS}]"], check=True)

# An editable install adds a .pth file, which a running kernel does not re-read.
# Put src/ on sys.path so the import works in this session without a restart.
src_dir = f"{REPO_DIR}/src"
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

# Tracking store: SQLite database on Drive (accepted in ADR-001; the file: store is deprecated).
os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{DRIVE_ROOT}/mlflow/mlflow.db"
os.environ["AEL_DATA_DIR"] = f"{DRIVE_ROOT}/data"

commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("Code version (commit):", commit)
print("Data dir:", os.environ["AEL_DATA_DIR"])

import ai_experiment_lab
print("ai_experiment_lab", ai_experiment_lab.__version__, "imported OK")

## 1. Choose the file and the target

In [ ]:
from pathlib import Path
import pandas as pd

DATA_FILE = "my_dataset.csv"     # <-- file name inside Drive/ai-experiment-lab/data/
TARGET = "target"                # <-- the column to predict
DATASET_NAME = "my-dataset"      # used as the model name in the registry
TITLE = None                     # heading of the web page, e.g. "Loan default checker"

TASK = None                      # None = infer. Or "binary" | "multiclass" | "regression"
ID_COLUMNS = None                # None = detect automatically. [] = none. Or a list of column names to leave out
POSITIVE_LABEL = None            # binary only: the class you want to detect (None = a 'yes'-like value, else the rarer class)
MODELS = None                    # None = all three for the task. Or e.g. ("logistic_regression", "xgboost")
TUNE_TRIALS = 0                  # Optuna trials per model; 0 = no tuning (try 20 once the baseline looks sensible)
MAX_ROWS = None                  # e.g. 50000 to train on a random sample of a very large file
USE_MLFLOW = True                # log the run and register the model

path = Path(DRIVE_ROOT) / "data" / DATA_FILE
df = pd.read_csv(path)
print(path.name, df.shape)
df.head()

## 2. Look before training: what will the platform do with this file?

In [ ]:
from ai_experiment_lab.automl import build_schema, DatasetError

try:
    schema, X, y, coerce, notes = build_schema(df, TARGET, TASK, ID_COLUMNS, POSITIVE_LABEL)
except DatasetError:
    print("Cannot use this file as it is. Fix the setting named in the message below and re-run step 1.")
    raise

print("task:", schema.task, "| target:", schema.target, "| rows:", len(X), "| features:", len(schema.features))
if schema.classes:
    print("classes:", schema.classes, "| positive:", schema.positive_label)
print("left out as identifiers:", schema.excluded)
for n in notes:
    print("note:", n)
pd.DataFrame([{k: v for k, v in f.to_dict().items() if k in ("name", "kind", "nullable", "min", "max", "allowed")}
              for f in schema.features])

Check the table above. If an ID column was **not** detected, or a useful column was dropped, set `ID_COLUMNS` in step 1 and re-run. A column that *is derived from the target* (for example a `status` that only exists after the customer left) must be removed too: the platform cannot know about that, and it would make the scores look perfect and the model useless.

## 3. Train and compare

In [ ]:
from ai_experiment_lab.automl import train_dataset

result = train_dataset(
    df, TARGET, task=TASK, id_columns=ID_COLUMNS, positive_label=POSITIVE_LABEL,
    models=MODELS, tune_trials=TUNE_TRIALS, max_rows=MAX_ROWS, progress=print)

print(f"\nBest by cross-validated {result.selection_metric}: {result.best}  ({result.seconds:.0f} s)")
for n in result.notes:
    print("note:", n)
cols = [c for c in result.table.columns if c in ("model", f"cv_{result.selection_metric}", "cv_selection_std", "fit_seconds") or c.startswith("test_")]
result.table[cols].round(4)

Read the table like this: models are **ranked by the `cv_` column** (cross-validation on the training split). The `test_` columns were measured once on rows no model saw; they are the honest estimate. If the top two are within `cv_selection_std` of each other, they are tied: prefer the simpler one.

**Be suspicious of near-perfect scores** (F1 or R^2 above about 0.99 on real-world data). It usually means a column leaks the answer, or the data is synthetic and rule-generated.

In [ ]:
print("Test-set metrics of the selected model (scored once):")
for k, v in result.test_metrics.items():
    print(f"  {k:>18}: {v:.4f}")

## 4. Optional: log to MLflow and register the model

In [ ]:
mlflow_info = None
if USE_MLFLOW:
    from ai_experiment_lab.automl.tracking import log_and_register
    from ai_experiment_lab.data.ingestion import file_sha256

    mlflow_info = log_and_register(result, DATASET_NAME, file_sha256(path))
    print(mlflow_info)

## 5. Export for the API

In [ ]:
import shutil
from ai_experiment_lab.automl import export_result
from ai_experiment_lab.data.ingestion import file_sha256

version = (mlflow_info or {}).get("version") or 1
out = export_result(result, f"{REPO_DIR}/models/champion", DATASET_NAME, version=version,
                    dataset_sha256=file_sha256(path), title=TITLE)
# keep a copy on Drive too
backup = Path(DRIVE_ROOT) / "exports" / f"{DATASET_NAME}-v{version}"
shutil.copytree(out, backup, dirs_exist_ok=True)
print("exported to", out, "\nbackup on Drive:", backup)

## 6. Deploy it

The two files in `models/champion/` are the whole model. To put it online (replaces the model currently served):

1. Download `models/champion/` (cell below) or copy it from the Drive backup into your repo at `models/champion/`.
2. Set the library versions printed below as the `ARG` defaults at the top of the `Dockerfile` (the image must match the training environment).
3. In VS Code: `pytest -q`, `python scripts/check_model_export.py`, then `git add models/champion Dockerfile`, commit and push.
4. Render: **Manual Deploy**, or GitHub **Actions > Redeploy**. Then `python scripts/smoke_test_api.py <your-url> --wait 120`.

In [ ]:
import json
meta = json.loads((Path(out) / "metadata.json").read_text())
libs = meta["libraries"]
print("In the Dockerfile set:")
print("  ARG SKLEARN_VERSION=" + libs.get("scikit-learn", "?"))
if any(t.startswith("xgboost.") for t in meta.get("skops_trusted_types", [])):
    print("  ARG XGBOOST_VERSION=" + libs.get("xgboost", "?"))
else:
    print("  (XGBOOST_VERSION can stay empty: the selected model does not use XGBoost)")

try:
    from google.colab import files
    shutil.make_archive("/content/champion", "zip", out)
    files.download("/content/champion.zip")
except ImportError:
    pass

## 7. Later: is the incoming data still like the training data?

After the API has served traffic with `PREDICTION_LOG` set, compare the logged inputs with this training CSV (works for any model exported here):

```
python scripts/check_drift.py --data data/raw/<your file>.csv --model-dir models/champion --log data/predictions/predictions.jsonl --out reports/drift.json
```

A `drift` verdict means the inputs moved; see `docs/retraining.md` for how to decide whether to retrain and when a new model may replace the old one (the promotion metric is ROC AUC for binary, balanced accuracy for multi-class and R^2 for regression).